# HybXAI-Sepsis v3 — Figures for Paper
Reproduces all manuscript figures from saved results artifacts.

**Run this notebook** after training (or use the pre-saved artifacts in ).

In [ ]:
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (roc_curve, precision_recall_curve,
    roc_auc_score, average_precision_score, f1_score, confusion_matrix,
    ConfusionMatrixDisplay)
plt.rcParams.update({"figure.dpi": 150, "font.size": 11})

# ── Load artifacts ──────────────────────────────────────────────
ROOT = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..")
FIGS = os.path.join(ROOT, "results", "figures")
TABS = os.path.join(ROOT, "results", "tables")

y_test       = np.load(os.path.join(ROOT, "results", "y_test.npy"))
hybxai_probs = np.load(os.path.join(ROOT, "results", "hybxai_probs_test.npy"))
y_eicu       = np.load(os.path.join(ROOT, "results", "y_eicu_eval.npy"))
eicu_probs   = np.load(os.path.join(ROOT, "results", "eicu_probs_recal.npy"))
t1 = pd.read_csv(os.path.join(TABS, "table1_performance.csv"))
t2 = pd.read_csv(os.path.join(TABS, "table2_horizons.csv"))
t3 = pd.read_csv(os.path.join(TABS, "table3_ablation.csv"))

print("Artifacts loaded.")
print(f"MIMIC-IV test: {len(y_test):,} stays | AUROC = {roc_auc_score(y_test, hybxai_probs):.4f}")
print(f"eICU eval:     {len(y_eicu):,} stays | AUROC = {roc_auc_score(y_eicu, eicu_probs):.4f}")


## Figure 2 — ROC & Precision-Recall Curves

In [ ]:
PALETTE = {
    "MEWS (Rule-based)":           "#666666",
    "Logistic Regression":         "#95A5A6",
    "Random Forest":               "#3498DB",
    "LightGBM":                    "#9B59B6",
    "BiLSTM + Attention":          "#E67E22",
    "XGBoost-Only":                "#1ABC9C",
    "HybXAI-Sepsis v3 (Proposed)": "#E74C3C",
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for _, row in t1.iterrows():
    name = row["Model"]; auroc = row["AUROC"]; auprc = row["AUPRC"]
    c = PALETTE.get(name, "#333333"); lw = 2.5 if "v3" in name else 1.5
    if name == "HybXAI-Sepsis v3 (Proposed)": probs = hybxai_probs
    else: continue
    fpr, tpr, _ = roc_curve(y_test, probs)
    axes[0].plot(fpr, tpr, color=c, lw=lw, label=f"{name} (AUROC={auroc})")
    pr, rc, _ = precision_recall_curve(y_test, probs)
    axes[1].plot(rc, pr, color=c, lw=lw, label=f"{name} (AUPRC={auprc})")

axes[0].plot([0,1],[0,1],"k--",alpha=0.4); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("ROC Curves — MIMIC-IV Test Set"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curves — MIMIC-IV Test Set"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
fig.suptitle("HybXAI-Sepsis v3 — Discrimination Performance", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGS, "fig2_roc_prc_curves.png"), dpi=200, bbox_inches="tight")
plt.show(); print("Fig 2 saved.")


## Figure 3 — Prediction Horizon Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("HybXAI-Sepsis v3 — Prediction Horizon Analysis", fontsize=13, fontweight="bold")
hs = ["H=3h","H=6h","H=12h"]; cols = ["#2ECC71","#3498DB","#E74C3C"]
for ax, metric, title in zip(axes, ["AUROC","F1","Sens"],
                               ["AUROC vs Lead Time","F1-Score vs Lead Time","Sensitivity vs Lead Time"]):
    vals = t2[metric].values
    bars = ax.bar(hs, vals, color=cols, alpha=0.85, edgecolor="black")
    ax.set_ylim(0.85, 1.01); ax.set_title(title); ax.set_ylabel(metric); ax.grid(axis="y",alpha=0.3)
    for bar,v in zip(bars,vals):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.002, f"{v:.3f}", ha="center",
                va="bottom", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGS,"fig3_horizon_analysis.png"),dpi=200,bbox_inches="tight")
plt.show(); print("Fig 3 saved.")


## Figure 4 — Ablation Study

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
cols = ["#2ECC71" if "Full" in c else "#E74C3C" for c in t3["Configuration"]]
bars = ax.bar(t3["Configuration"], t3["AUROC"], color=cols, alpha=0.85, edgecolor="black")
ax.set_ylim(t3["AUROC"].min()-0.05, 1.01); ax.set_ylabel("AUROC")
ax.set_title("HybXAI-Sepsis v3 — Ablation Study"); ax.tick_params(axis="x", rotation=20)
for bar, v in zip(bars, t3["AUROC"]):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.002, f"{v:.4f}",
            ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.grid(axis="y",alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(FIGS,"fig4_ablation_study.png"),dpi=200,bbox_inches="tight")
plt.show(); print("Fig 4 saved.")


## Figure 11 — Confusion Matrix

In [ ]:
from sklearn.metrics import roc_curve as _rc
fpr_,tpr_,thr_ = _rc(y_test, hybxai_probs)
thr_opt = thr_[np.argmax(tpr_-fpr_)]
y_pred  = (hybxai_probs >= thr_opt).astype(int)
cm = confusion_matrix(y_test, y_pred)
tn,fp,fn,tp = cm.ravel()
fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.suptitle("HybXAI-Sepsis v3 — Confusion Matrix at Youden-Optimal Threshold",
             fontsize=13, fontweight="bold")
ConfusionMatrixDisplay(cm,display_labels=["Non-Sepsis","Sepsis"]).plot(
    ax=axes[0],colorbar=False,cmap="Blues")
axes[0].set_title(f"Counts | Thr={thr_opt:.4f}")
cm_n = cm.astype(float)/cm.sum(axis=1,keepdims=True)
ConfusionMatrixDisplay(cm_n.round(3),display_labels=["Non-Sepsis","Sepsis"]).plot(
    ax=axes[1],colorbar=False,cmap="Blues")
axes[1].set_title(f"Row-Normalised | Sens={tp/(tp+fn+1e-8):.4f}, Spec={tn/(tn+fp+1e-8):.4f}")
plt.tight_layout()
plt.savefig(os.path.join(FIGS,"fig11_confusion_matrix.png"),dpi=200,bbox_inches="tight")
plt.show(); print("Fig 11 saved.")


## Figure — Cross-Institutional Generalisation

In [ ]:
mimic_auroc = roc_auc_score(y_test, hybxai_probs)
eicu_auroc  = roc_auc_score(y_eicu, eicu_probs)
ams_auroc   = 0.7680  # reported in paper (AMS-style synthetic cohort)

fig, axes = plt.subplots(1,2,figsize=(14,6))
fig.suptitle("HybXAI-Sepsis v3 — Cross-Institutional Generalisation",fontsize=13,fontweight="bold")
cohorts = ["MIMIC-IV\n(Internal)","eICU-CRD\n(US multi-site)","AMS-style\n(European)"]
aurocs  = [mimic_auroc, eicu_auroc, ams_auroc]
c_list  = ["#2ECC71","#3498DB","#9B59B6"]
bars = axes[0].bar(cohorts,aurocs,color=c_list,alpha=0.85,edgecolor="black")
axes[0].set_ylim(0.7,1.0); axes[0].set_ylabel("AUROC"); axes[0].set_title("Cross-Cohort AUROC")
axes[0].axhline(0.90,ls="--",color="gray",alpha=0.5,label="AUROC=0.90 reference")
axes[0].legend(); axes[0].grid(axis="y",alpha=0.3)
for bar,v in zip(bars,aurocs):
    axes[0].text(bar.get_x()+bar.get_width()/2,v+0.003,f"{v:.4f}",
                 ha="center",va="bottom",fontweight="bold")
ref_names  = ["Desautels\n(InSight)","Mao et al.\n(GB)","Du et al.\n(BiLSTM)",
              "Shukla &\nMarlin (mTAN)","HybXAI-Sepsis\nv3 (eICU)"]
ref_aurocs = [0.880,0.863,0.893,0.854,eicu_auroc]
ref_cols   = ["#95A5A6"]*4 + ["#E74C3C"]
axes[1].bar(ref_names,ref_aurocs,color=ref_cols,alpha=0.85,edgecolor="black")
axes[1].set_ylim(0.80,0.95); axes[1].set_ylabel("AUROC")
axes[1].set_title("Comparison vs Reference Models (eICU/multi-site)"); axes[1].grid(axis="y",alpha=0.3)
for i,(v,c) in enumerate(zip(ref_aurocs,ref_cols)):
    axes[1].text(i,v+0.001,f"{v:.3f}",ha="center",va="bottom",
                 fontweight="bold",color=c if c!="#95A5A6" else "black")
plt.tight_layout()
plt.savefig(os.path.join(FIGS,"fig_cross_institutional.png"),dpi=200,bbox_inches="tight")
plt.show(); print("Cross-institutional figure saved.")


## All figures complete
All figures are saved to .